# SAFE-Alert — Train BTCUSDT 4h

Notebook độc lập. Dùng 32 news candidates, lookback 96 giờ và model Top-5. Scalar features và market bars được tạo trong workspace riêng để không ghi đè dữ liệu 1h.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess
HORIZON='4h'; SYMBOL='BTCUSDT'; EPOCHS=20; N_FOLDS=4; BATCH_SIZE=8; RUN_SMOKE_TEST=True
candidates=[Path.cwd(),Path('/kaggle/working/CQ_2025-AI_Sentiment_Support-Private/services/ai-service')]
if Path('/kaggle/input').exists(): candidates += [p.parents[3] for p in Path('/kaggle/input').glob('**/app/v2/pipelines/train_safe_alert.py')]
SERVICE_ROOT=next((p.resolve() for p in candidates if (p/'app/v2/pipelines/train_safe_alert.py').exists()),None)
assert SERVICE_ROOT, 'Không tìm thấy services/ai-service.'
SOURCE_DATA=SERVICE_ROOT/'training_data/v2'
WORK=Path('/kaggle/working/data_4h') if Path('/kaggle/working').exists() else SERVICE_ROOT/'training_data/notebook_4h'
ARTIFACTS=Path('/kaggle/working/safe_alert_4h') if Path('/kaggle/working').exists() else SERVICE_ROOT/'artifacts/safe_alert_4h'
BASE_CONFIG=SERVICE_ROOT/'app/v2/pipelines/train_config_research_best.yaml'
print(SERVICE_ROOT, SOURCE_DATA, WORK, ARTIFACTS, sep='\n')

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','ta==0.11.0','vaderSentiment==3.3.2','PyYAML>=6.0.3,<6.1'],check=True)
import numpy as np, pandas as pd, torch, yaml
assert torch.cuda.is_available(), 'Hãy bật GPU accelerator.'
WORK.mkdir(parents=True,exist_ok=True)
needed=['BTCUSDT_1m_ohlcv.csv','BTCUSDT_5m_ohlcv.csv','BTCUSDT_15m_ohlcv.csv','BTCUSDT_1h_ohlcv.csv','BTCUSDT_4h_ohlcv.csv','articles_max.csv','btcusdt_article_embeddings_max.npy','article_factor_labels.npy','article_entity_sentiment.npy','article_novelty.npy','market_bars_4h.npz']
for name in needed:
    src=SOURCE_DATA/name; assert src.exists(),f'Thiếu {src}'
    dst=WORK/name
    if not dst.exists():
        try: os.symlink(src,dst)
        except OSError: shutil.copy2(src,dst)

In [ ]:
sys.path.insert(0,str(SERVICE_ROOT/'app/v2/pipelines'))
from precompute_market_features import precompute_all_features
feature_path=WORK/'features_precomputed.npy'
if not feature_path.exists(): precompute_all_features(WORK/'BTCUSDT_4h_ohlcv.csv',feature_path)
bars=WORK/'market_bars_4h.npz'
if not bars.exists():
    subprocess.run([sys.executable,'app/v2/pipelines/precompute_market_bars.py','--data-dir',str(WORK),'--symbol',SYMBOL,'--decision-horizon','4h','--out',str(bars)],cwd=SERVICE_ROOT,check=True)
n=sum(1 for _ in open(WORK/'BTCUSDT_4h_ohlcv.csv',encoding='utf-8-sig'))-1
assert np.load(feature_path,mmap_mode='r').shape==(n,63)
with np.load(bars) as z: assert all(z[k].shape==(n,20,10) for k in z.files)
print('4h rows/features/bars OK:',n)

In [ ]:
cfg=yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
cfg.update({'horizon':'4h','articles_per_candle':32,'lookback_hours':96,'epochs':EPOCHS,'n_folds':N_FOLDS,'embargo_steps':48})
cfg.setdefault('top_k',{})['K_4h']=5
config_path=WORK/'train_config_4h.yaml'; config_path.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding='utf-8')
cmd_base=[sys.executable,'app/v2/pipelines/train_safe_alert.py','--config',str(config_path),'--symbol',SYMBOL,'--horizon','4h','--data_path',str(WORK),'--embeddings_path',str(WORK),'--batch_size',str(BATCH_SIZE),'--walk_forward','--use_bar_sequences']
print(config_path.read_text(encoding='utf-8')[:1200])

In [ ]:
if RUN_SMOKE_TEST:
    subprocess.run(cmd_base+['--epochs','2','--n_folds','1','--artifact_dir',str(ARTIFACTS.parent/'smoke_4h')],cwd=SERVICE_ROOT,check=True)
ARTIFACTS.mkdir(parents=True,exist_ok=True)
subprocess.run(cmd_base+['--epochs',str(EPOCHS),'--n_folds',str(N_FOLDS),'--artifact_dir',str(ARTIFACTS)],cwd=SERVICE_ROOT,check=True)

In [ ]:
zip_path=shutil.make_archive(str(ARTIFACTS),'zip',ARTIFACTS)
print('Download artifact:',zip_path)
print('\n'.join(str(p.relative_to(ARTIFACTS)) for p in sorted(ARTIFACTS.rglob('*')) if p.is_file()))